In [1]:
import pandas as pd

df_2009_2010 = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name="Year 2009-2010")
df_2010_2011 = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name="Year 2010-2011")
df = pd.concat([df_2009_2010, df_2010_2011], ignore_index=True)

print("Start-Shape:", df.shape)

df["is_cancellation"] = df["Invoice"].astype(str).str.startswith("C")
print("Anzahl Stornos:", df["is_cancellation"].sum())

# Stimmen negative Menge und Storno-Kennzeichnung überein?
negative_qty_no_c = df[(df["Quantity"] < 0) & (~df["is_cancellation"])]
print("Negative Menge OHNE 'C'-Präfix:", len(negative_qty_no_c))
negative_qty_no_c.head(10)


negative_price = df[df["Price"] < 0]
print("Anzahl Zeilen mit negativem Preis:", len(negative_price))
negative_price[["Invoice", "StockCode", "Description", "Quantity", "Price"]].head(10)

anzahl_duplikate = df.duplicated().sum()
print("Anzahl exakter Duplikate:", anzahl_duplikate)

Start-Shape: (1067371, 8)
Anzahl Stornos: 19494
Negative Menge OHNE 'C'-Präfix: 3457
Anzahl Zeilen mit negativem Preis: 5
Anzahl exakter Duplikate: 34335


In [2]:
print(negative_qty_no_c["StockCode"].value_counts().head(15))
print()
print(negative_qty_no_c["Description"].value_counts().head(15))

StockCode
22423     12
46000M     6
82494L     6
22719      6
46000S     5
84016      5
85017A     5
47566B     5
20852      5
85175      5
21830      5
71477      4
21768      4
84559D     4
84990      4
Name: count, dtype: int64

Description
check                     123
damages                    84
?                          83
damaged                    78
missing                    27
sold as set on dotcom      20
Damaged                    17
smashed                     9
thrown away                 9
Unsaleable, destroyed.      9
dotcom                      8
damages?                    7
??                          7
crushed                     6
given away                  6
Name: count, dtype: int64


In [3]:
negative_price[["Invoice", "StockCode", "Description", "Quantity", "Price"]]

,Invoice,StockCode,Description,Quantity,Price
179403,A506401,B,Adjust bad debt,1,-53594.36
276274,A516228,B,Adjust bad debt,1,-44031.79
403472,A528059,B,Adjust bad debt,1,-38925.87
825444,A563186,B,Adjust bad debt,1,-11062.06
825445,A563187,B,Adjust bad debt,1,-11062.06


In [4]:
fehlende_customer_id = negative_qty_no_c["Customer ID"].isna().sum()
print(f"{fehlende_customer_id} von {len(negative_qty_no_c)} Lagerkorrektur-Zeilen haben keine Customer ID")

3457 von 3457 Lagerkorrektur-Zeilen haben keine Customer ID


In [5]:
# Ausgangsdaten (falls noch nicht geladen)
print("Start:", df.shape)

# 1. Exakte Duplikate entfernen
df = df.drop_duplicates()
print("Nach Duplikate entfernen:", df.shape)

# 2. Negative Preise entfernen (Buchhaltungskorrekturen, keine echten Verkäufe)
df = df[df["Price"] >= 0]
print("Nach Entfernen negativer Preise:", df.shape)

# 3. Zeilen ohne Customer ID entfernen (nicht kundenbezogen zuordenbar)
df = df.dropna(subset=["Customer ID"])
print("Nach Entfernen fehlender Customer ID:", df.shape)

# 4. Customer ID zu Integer casten (kein NaN mehr -> sauberer Typ)
df["Customer ID"] = df["Customer ID"].astype(int)

# 5. Storno-Kennzeichnung final setzen (für spätere Analysen nützlich)
df["is_cancellation"] = df["Invoice"].astype(str).str.startswith("C")

# 6. Umsatzspalte berechnen (bei Stornos automatisch negativ -> mindert Netto-Umsatz korrekt)
df["Revenue"] = df["Quantity"] * df["Price"]

print("\nFinale Form:", df.shape)
df.head()

Start: (1067371, 9)
Nach Duplikate entfernen: (1033036, 9)
Nach Entfernen negativer Preise: (1033031, 9)
Nach Entfernen fehlender Customer ID: (797885, 9)

Finale Form: (797885, 10)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,is_cancellation,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,False,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,False,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,False,30.0


In [6]:
df.to_csv("../data/processed/online_retail_clean.csv", index=False)
print("Bereinigter Datensatz gespeichert:", df.shape)

Bereinigter Datensatz gespeichert: (797885, 10)
